<a href="https://colab.research.google.com/github/RajeshworM/Yield_Modelling_Automation/blob/main/SoybeanYield_MP_Cont.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

# =========================
# 1. LOAD & SORT
# =========================

from google.colab import files
uploaded = files.upload()

df = pd.read_excel(list(uploaded.keys())[0])

df = df.sort_values(
    ["district_name", "year"]
).reset_index(drop=True)

# =========================
# 2. MODEL COEFFICIENTS
# SOYABEAN MADHYA PRADESH
# =========================

b = {

    "ln_rel_price_sy_frtptd": 1.119523,

    "ln_rv_sstd_irr": 0.9194725,

    "irr_ratio": -0.4958874,

    "ln_rf_veg_total": 1.633242,

    "ln_rf_veg_total_sq": -0.1433855,

    "ln_tmin_rep_avg": -5.530265,

    "ln_rh_rep_mean": -1.055298,

    "coldstress_rep_total": -0.4073767,

    "htstress_sow_total": -0.0122781,

    "excess10rf_sow_total": -0.0223859,

    "excess20rf_mthv_total": -0.3360013
}

intercept = 20.66163

# =========================
# 3. YEAR EFFECTS
# =========================

year_effects = {

    2011: -0.0795984,

    2012: 0.5762769,

    2013: -0.1446255,

    2014: 0.4388608,

    2015: 0.0154087,

    2016: 0.5160189,

    2017: -0.0736344,

    2018: -0.0946964,

    2019: 0.0953137,

    2020: -0.4409822,

    2021: 0.1876004,

    2022: 0.1039048,

    2023: -0.2576261,

    2024: 0,

    2025: 0
}

# =========================
# 4. VARIABLE GROUPS
# =========================

stress_vars = [

    "coldstress_rep_total",

    "htstress_sow_total",

    "excess10rf_sow_total",

    "excess20rf_mthv_total"
]

nonstress_vars = [
    v for v in b
    if v not in stress_vars
]

# =========================
# 5. 3-YEAR MOVING AVERAGE
# BENCHMARK
# =========================

df = df.sort_values(
    ["district_name", "year"]
)

for v in nonstress_vars:

    df["avg_" + v] = (

        df.groupby("district_name")[v]

          .transform(

              lambda x:
              x.shift(1).rolling(
                  window=3,
                  min_periods=1
              ).mean()

          )
    )

# Stress benchmark = 0

for v in stress_vars:

    df["avg_" + v] = 0

# =========================
# 6. DIFFERENCE FROM NORMAL
# =========================

for v in b:

    df["diff_" + v] = (

        df[v] - df["avg_" + v]

    )

# =========================
# 7. IMPACTS
# =========================

for v in b:

    df["impact_" + v] = (

        b[v] * df["diff_" + v]

    )

# =========================
# 7.5 WATER GROUP
# =========================

df["impact_Water_group"] = (

    df["impact_ln_rv_sstd_irr"] +

    df["impact_irr_ratio"]

)

# =========================
# 7.6 RAINFALL GROUP
# =========================

df["impact_Rainfall_group"] = (

    df["impact_ln_rf_veg_total"] +

    df["impact_ln_rf_veg_total_sq"]

)

# =========================
# 7.7 EXCESS RAIN GROUP
# =========================

df["impact_ExcessRain_group"] = (

    df["impact_excess10rf_sow_total"] +

    df["impact_excess20rf_mthv_total"]

)

# =========================
# 8. YEAR EFFECT
# =========================

df["year_effect"] = (

    df["year"]

      .map(year_effects)

      .fillna(0)

)

# =========================
# 9. PREDICTED YIELD
# =========================

df["ln_yield_sy_pred"] = intercept

for v in b:

    df["ln_yield_sy_pred"] += (

        b[v] * df[v]

    )

# Add year FE

df["ln_yield_sy_pred"] += (

    df["year_effect"]

)

# Predicted yield

df["yield_pred"] = np.exp(

    df["ln_yield_sy_pred"]

)

# =========================
# 10. ACTUAL YIELD
# =========================

df["yield_actual"] = np.exp(

    df["ln_yield_sy"]

)

# =========================
# 11. NORMAL YIELD
# (3-YEAR MOVING AVERAGE)
# =========================

df["yield_normal"] = (

    df.groupby("district_name")["yield_actual"]

      .transform(

          lambda x:
          x.shift(1).rolling(
              window=3,
              min_periods=1
          ).mean()

      )
)

# =========================
# 12. % DEVIATION
# =========================

df["yield_%dev_from_normal"] = np.where(

    df["yield_normal"].notna(),

    (

        (df["yield_pred"] - df["yield_normal"])

        / df["yield_normal"]

    ) * 100,

    np.nan
)

# =========================
# 13. DIRECTION CHECK
# =========================

df["actual_change"] = (

    df.groupby("district_name")["ln_yield_sy"]

      .diff()

)

df["pred_change"] = (

    df.groupby("district_name")["ln_yield_sy_pred"]

      .diff()

)

df["actual_dir"] = np.nan
df["pred_dir"] = np.nan

df.loc[
    df["actual_change"] > 0,
    "actual_dir"
] = "Up"

df.loc[
    df["actual_change"] < 0,
    "actual_dir"
] = "Down"

df.loc[
    df["pred_change"] > 0,
    "pred_dir"
] = "Up"

df.loc[
    df["pred_change"] < 0,
    "pred_dir"
] = "Down"

df["direction_match"] = np.nan

mask = (

    df["actual_dir"].notna()

    &

    df["pred_dir"].notna()

)

df.loc[
    mask,
    "direction_match"
] = np.where(

    df.loc[mask, "actual_dir"]

    ==

    df.loc[mask, "pred_dir"],

    "Correct",

    "Wrong"
)

# =========================
# 14. LABEL MAP
# =========================

label_map = {

    "ln_rel_price_sy_frtptd":
        "Price relative to FRTPTD",

    "ln_rv_sstd_irr":
        "Reservoir & Surface Irrigation",

    "irr_ratio":
        "Irrigation Ratio",

    "ln_rf_veg_total":
        "Rainfall - Veg",

    "ln_rf_veg_total_sq":
        "Rainfall-Veg Quadratic",

    "ln_tmin_rep_avg":
        "Minimum Temperature-Rep",

    "ln_rh_rep_mean":
        "Humidity-Rep",

    "coldstress_rep_total":
        "Cold Stress-Rep",

    "htstress_sow_total":
        "Heat Stress-Sow",

    "excess10rf_sow_total":
        "Excess Rainfall-Sow",

    "excess20rf_mthv_total":
        "Extreme Rainfall -Mat)",

    "Water_group":
        "Water-Management",

    "Rainfall_group":
        "Rainfall Environment",

    "ExcessRain_group":
        "Stress-Excess Rainfall- SowMat",

    "Year_effect":
        "Year Specific Effect"
}

# =========================
# 15. MODEL EXPLANATION
# =========================

def explain(row):

    ydev = row["yield_%dev_from_normal"]

    out = {

        "YIELD-MAGNITUDE": np.nan,

        "Contributor-1": np.nan,

        "Contributor-2": np.nan,

        "Contributor-3": np.nan,

        "Counter-1": np.nan
    }

    if pd.isna(ydev):

        return pd.Series(out)

    # Individual drivers
    # excluding grouped variables

    drivers = {

        v: row["impact_" + v]

        for v in b

        if v not in [

            "ln_rv_sstd_irr",

            "irr_ratio",

            "ln_rf_veg_total",

            "ln_rf_veg_total_sq",

            "excess10rf_sow_total",

            "excess20rf_mthv_total"
        ]
    }

    # Add grouped effects

    drivers["Water_group"] = (

        row["impact_Water_group"]

    )

    drivers["Rainfall_group"] = (

        row["impact_Rainfall_group"]

    )

    drivers["ExcessRain_group"] = (

        row["impact_ExcessRain_group"]

    )

    # Add year FE

    drivers["Year_effect"] = (

        row["year_effect"]

    )

    # =====================
    # LOW YIELD
    # =====================

    if ydev < 0:

        out["YIELD-MAGNITUDE"] = "LOW"

        neg = {

            k: v

            for k, v in drivers.items()

            if v < 0
        }

        pos = {

            k: v

            for k, v in drivers.items()

            if v > 0
        }

        top_neg = sorted(

            neg.items(),

            key=lambda x: x[1]

        )[:3]

        for i, (var, _) in enumerate(

            top_neg, 1

        ):

            out[f"Contributor-{i}"] = (

                label_map.get(var, var)

            )

        if pos:

            counter = max(

                pos.items(),

                key=lambda x: x[1]

            )[0]

            out["Counter-1"] = (

                label_map.get(counter, counter)

            )

    # =====================
    # HIGH YIELD
    # =====================

    else:

        out["YIELD-MAGNITUDE"] = "HIGH"

        pos = {

            k: v

            for k, v in drivers.items()

            if v > 0
        }

        neg = {

            k: v

            for k, v in drivers.items()

            if v < 0
        }

        top_pos = sorted(

            pos.items(),

            key=lambda x: x[1],

            reverse=True

        )[:3]

        for i, (var, _) in enumerate(

            top_pos, 1

        ):

            out[f"Contributor-{i}"] = (

                label_map.get(var, var)

            )

        if neg:

            counter = min(

                neg.items(),

                key=lambda x: x[1]

            )[0]

            out["Counter-1"] = (

                label_map.get(counter, counter)

            )

    return pd.Series(out)

# Apply explanation

df[[
    "YIELD-MAGNITUDE",
    "Contributor-1",
    "Contributor-2",
    "Contributor-3",
    "Counter-1"
]] = df.apply(

    explain,

    axis=1
)

# =========================
# 16. EXPORT
# =========================

output_file = (

    "Soyabean_MP_3yrMA_Benchmark_Attribution.xlsx"

)

df.to_excel(

    output_file,

    index=False

)

files.download(output_file)

print(
    "✅ Soyabean MP 3-year MA benchmark attribution model completed successfully."
)

Saving Soybean_mp_proc.xlsx to Soybean_mp_proc.xlsx


/tmp/ipykernel_6975/3314450265.py:314: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Up' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[
/tmp/ipykernel_6975/3314450265.py:324: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Up' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[
/tmp/ipykernel_6975/3314450265.py:346: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['Correct' 'Wrong' 'Correct' 'Correct' 'Wrong' 'Correct' 'Correct'
 'Correct' 'Wrong' 'Wrong' 'Correct' 'Correct' 'Correct' 'Correct'
 'Correct' 'Correct' 'Wrong' 'Correct' 'Correct' 'Correct' 'Correct'
 'Correct' 'Wrong' 'Correct' 'Correct' 'Correct' 'Correct' 'Correct'
 'Correct' 'Wrong'

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Soyabean MP 3-year MA benchmark attribution model completed successfully.


***Revised added district fixed effect***

In [4]:
import pandas as pd
import numpy as np

# =========================
# 1. LOAD & SORT
# =========================

from google.colab import files
uploaded = files.upload()

df = pd.read_excel(list(uploaded.keys())[0])

df = df.sort_values(
    ["district_name", "year"]
).reset_index(drop=True)

# =========================
# 2. MODEL COEFFICIENTS
# =========================

b = {

    "ln_rel_price_sy_frtptd": 1.119523,
    "ln_rv_sstd_irr": 0.9194725,
    "irr_ratio": -0.4958874,
    "ln_rf_veg_total": 1.633242,
    "ln_rf_veg_total_sq": -0.1433855,
    "ln_tmin_rep_avg": -5.530265,
    "ln_rh_rep_mean": -1.055298,
    "coldstress_rep_total": -0.4073767,
    "htstress_sow_total": -0.0122781,
    "excess10rf_sow_total": -0.0223859,
    "excess20rf_mthv_total": -0.3360013
}

intercept = 20.66163

# =========================
# 3. YEAR EFFECTS
# =========================

year_effects = {

    2011: -0.0795984,
    2012: 0.5762769,
    2013: -0.1446255,
    2014: 0.4388608,
    2015: 0.0154087,
    2016: 0.5160189,
    2017: -0.0736344,
    2018: -0.0946964,
    2019: 0.0953137,
    2020: -0.4409822,
    2021: 0.1876004,
    2022: 0.1039048,
    2023: -0.2576261,
    2024: 0,
    2025: 0
}

# =========================
# 4. VARIABLE GROUPS
# =========================

stress_vars = [
    "coldstress_rep_total",
    "htstress_sow_total",
    "excess10rf_sow_total",
    "excess20rf_mthv_total"
]

nonstress_vars = [v for v in b if v not in stress_vars]

# =========================
# 5. 3-YEAR MOVING AVERAGE
# =========================

df = df.sort_values(["district_name", "year"])

for v in nonstress_vars:
    df["avg_" + v] = (
        df.groupby("district_name")[v]
          .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
    )

for v in stress_vars:
    df["avg_" + v] = 0

# =========================
# 6. DIFFERENCE FROM NORMAL
# =========================

for v in b:
    df["diff_" + v] = df[v] - df["avg_" + v]

# =========================
# 7. IMPACTS
# =========================

for v in b:
    df["impact_" + v] = b[v] * df["diff_" + v]

# =========================
# 7.5 GROUPED EFFECTS
# =========================

df["impact_Water_group"] = (
    df["impact_ln_rv_sstd_irr"] +
    df["impact_irr_ratio"]
)

df["impact_Rainfall_group"] = (
    df["impact_ln_rf_veg_total"] +
    df["impact_ln_rf_veg_total_sq"]
)

df["impact_ExcessRain_group"] = (
    df["impact_excess10rf_sow_total"] +
    df["impact_excess20rf_mthv_total"]
)

# =========================
# 8. YEAR EFFECT
# =========================

df["year_effect"] = df["year"].map(year_effects).fillna(0)

# =========================
# 9. STRUCTURAL MODEL
# =========================

df["ln_yield_struct"] = intercept

for v in b:
    df["ln_yield_struct"] += b[v] * df[v]

df["ln_yield_struct"] += df["year_effect"]

# =========================
# 10. DISTRICT FIXED EFFECT
# =========================

df["residual"] = df["ln_yield_sy"] - df["ln_yield_struct"]

district_fe = df.groupby("district_name")["residual"].mean()

df = df.merge(
    district_fe.rename("district_fe"),
    on="district_name",
    how="left"
)

# =========================
# 11. FINAL PREDICTION
# =========================

df["ln_yield_sy_pred"] = (
    df["ln_yield_struct"] + df["district_fe"]
)

df["yield_pred"] = np.exp(df["ln_yield_sy_pred"])

# =========================
# 12. ACTUAL YIELD
# =========================

df["yield_actual"] = np.exp(df["ln_yield_sy"])

# =========================
# 13. NORMAL YIELD
# =========================

df["yield_normal"] = (
    df.groupby("district_name")["yield_actual"]
      .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
)

# =========================
# 14. % DEVIATION
# =========================

df["yield_%dev_from_normal"] = np.where(
    df["yield_normal"].notna(),
    (df["yield_pred"] - df["yield_normal"]) / df["yield_normal"] * 100,
    np.nan
)

# =========================
# 15. DIRECTION CHECK (FIXED - NO NUMPY ISSUE)
# =========================

df["actual_change"] = df.groupby("district_name")["ln_yield_sy"].diff()
df["pred_change"] = df.groupby("district_name")["ln_yield_sy_pred"].diff()

df["actual_dir"] = np.nan
df["pred_dir"] = np.nan

df.loc[df["actual_change"] > 0, "actual_dir"] = "Up"
df.loc[df["actual_change"] < 0, "actual_dir"] = "Down"

df.loc[df["pred_change"] > 0, "pred_dir"] = "Up"
df.loc[df["pred_change"] < 0, "pred_dir"] = "Down"

mask = df["actual_dir"].notna() & df["pred_dir"].notna()

df["direction_match"] = np.nan

df.loc[mask, "direction_match"] = np.where(
    df.loc[mask, "actual_dir"] == df.loc[mask, "pred_dir"],
    "Correct",
    "Wrong"
)

# =========================
# 16. LABEL MAP
# =========================

label_map = {

    "ln_rel_price_sy_frtptd": "Price relative to FRTPTD",
    "ln_rv_sstd_irr": "Reservoir & Surface Irrigation",
    "irr_ratio": "Irrigation Ratio",
    "ln_rf_veg_total": "Rainfall- Veg",
    "ln_rf_veg_total_sq": "Rainfall-Veg (Quadratic)",
    "ln_tmin_rep_avg": "Min Temp-Rep",
    "ln_rh_rep_mean": "Humidity-Rep",
    "coldstress_rep_total": "Cold Stress",
    "htstress_sow_total": " Stress Heat-Sow",
    "excess10rf_sow_total": "Excess Rain-Sow",
    "excess20rf_mthv_total": "Excess Rain-Mat",
    "Water_group": "Water Management",
    "Rainfall_group": "Rainfall Effect-Veg",
    "ExcessRain_group": "Stress-Excess Rain-SowMat",
    "district_fe": "District Fixed Effect",
    "year_effect": "Year Effect"
}

# =========================
# 17. EXPLANATION ENGINE
# =========================

def explain(row):

    ydev = row["yield_%dev_from_normal"]

    out = {
        "YIELD-MAGNITUDE": np.nan,
        "Contributor-1": np.nan,
        "Contributor-2": np.nan,
        "Contributor-3": np.nan,
        "Counter-1": np.nan
    }

    if pd.isna(ydev):
        return pd.Series(out)

    drivers = {v: row["impact_" + v] for v in b if v not in [
        "ln_rv_sstd_irr",
        "irr_ratio",
        "ln_rf_veg_total",
        "ln_rf_veg_total_sq",
        "excess10rf_sow_total",
        "excess20rf_mthv_total"
    ]}

    drivers["Water_group"] = row["impact_Water_group"]
    drivers["Rainfall_group"] = row["impact_Rainfall_group"]
    drivers["ExcessRain_group"] = row["impact_ExcessRain_group"]
    drivers["Year_effect"] = row["year_effect"]
    drivers["District_FE"] = row["district_fe"]

    if ydev < 0:

        out["YIELD-MAGNITUDE"] = "LOW"

        neg = {k: v for k, v in drivers.items() if v < 0}
        pos = {k: v for k, v in drivers.items() if v > 0}

        top_neg = sorted(neg.items(), key=lambda x: x[1])[:3]

        for i, (var, _) in enumerate(top_neg, 1):
            out[f"Contributor-{i}"] = label_map.get(var, var)

        if pos:
            counter = max(pos.items(), key=lambda x: x[1])[0]
            out["Counter-1"] = label_map.get(counter, counter)

    else:

        out["YIELD-MAGNITUDE"] = "HIGH"

        pos = {k: v for k, v in drivers.items() if v > 0}
        neg = {k: v for k, v in drivers.items() if v < 0}

        top_pos = sorted(pos.items(), key=lambda x: x[1], reverse=True)[:3]

        for i, (var, _) in enumerate(top_pos, 1):
            out[f"Contributor-{i}"] = label_map.get(var, var)

        if neg:
            counter = min(neg.items(), key=lambda x: x[1])[0]
            out["Counter-1"] = label_map.get(counter, counter)

    return pd.Series(out)

df[[
    "YIELD-MAGNITUDE",
    "Contributor-1",
    "Contributor-2",
    "Contributor-3",
    "Counter-1"
]] = df.apply(explain, axis=1)

# =========================
# 18. EXPORT
# =========================

output_file = "Soyabean_MP_FE_Attribution.xlsx"

df.to_excel(output_file, index=False)

files.download(output_file)

print("✅ Model completed successfully (FE + Year + Attribution + Stable Direction Fix)")

Saving Soybean_mp_proc.xlsx to Soybean_mp_proc (3).xlsx


/tmp/ipykernel_4869/803950883.py:198: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Up' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[df["actual_change"] > 0, "actual_dir"] = "Up"
/tmp/ipykernel_4869/803950883.py:201: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Up' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[df["pred_change"] > 0, "pred_dir"] = "Up"
/tmp/ipykernel_4869/803950883.py:208: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['Correct' 'Wrong' 'Correct' 'Correct' 'Wrong' 'Correct' 'Correct'
 'Correct' 'Wrong' 'Wrong' 'Correct' 'Correct' 'Correct' 'Correct'
 'Correct' 'Correct' 'Wrong' 'Correct' 'Correct' 'Correct' 'Correct'
 'Co

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Model completed successfully (FE + Year + Attribution + Stable Direction Fix)
